# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 10
""").show()

# Load the daily performance data as a DuckDB view.
con.sql(f"""
CREATE OR REPLACE VIEW daily AS
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

# Check the date range so we know what historical data is available.
con.sql("""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS n
FROM daily
""").show()



┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│  min_date  │  max_date  │    n     │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │ 78835655 │
└────────────┴────────────┴──────────┘



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
# ============================================================
# SECTION 1 — SIGNAL CHECKS
# ============================================================

# ------------------------------------------------------------
# Signal 1: staleness
# ------------------------------------------------------------

print("SIGNAL 1 — STALENESS BUCKETS")

con.sql("""
WITH latest AS (
    SELECT MAX(report_date) AS decision_date
    FROM daily
),
content_last_seen AS (
    SELECT
        content_hash_id,
        MAX(report_date) AS last_observed
    FROM daily
    GROUP BY content_hash_id
),
bucketed AS (
    SELECT
        CASE
            WHEN DATE_DIFF('day', last_observed, decision_date) < 7
                THEN '<7 days'
            WHEN DATE_DIFF('day', last_observed, decision_date) < 30
                THEN '7-29 days'
            WHEN DATE_DIFF('day', last_observed, decision_date) < 60
                THEN '30-59 days'
            ELSE '60+ days'
        END AS staleness_bucket
    FROM content_last_seen
    CROSS JOIN latest
)
SELECT
    staleness_bucket,
    COUNT(*) AS n,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS pct
FROM bucketed
GROUP BY 1
ORDER BY
    CASE staleness_bucket
        WHEN '<7 days' THEN 1
        WHEN '7-29 days' THEN 2
        WHEN '30-59 days' THEN 3
        WHEN '60+ days' THEN 4
    END
""").show()


# ------------------------------------------------------------
# Signal 2: CTR versus position
# ------------------------------------------------------------

print("SIGNAL 2 — CTR BY POSITION BUCKET")

con.sql("""
WITH bucketed AS (
    SELECT
        CASE
            WHEN gsc_avg_position < 3 THEN '1-2.9'
            WHEN gsc_avg_position < 6 THEN '3-5.9'
            WHEN gsc_avg_position < 11 THEN '6-10.9'
            WHEN gsc_avg_position < 21 THEN '11-20.9'
            ELSE '21+'
        END AS position_bucket,

        CASE
            WHEN gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) < 0.01
                THEN '<1%'
            WHEN gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) < 0.03
                THEN '1-3%'
            WHEN gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) < 0.05
                THEN '3-5%'
            ELSE '5%+'
        END AS ctr_bucket

    FROM daily
    WHERE gsc_impressions > 0
      AND gsc_avg_position IS NOT NULL
)
SELECT
    position_bucket,
    ctr_bucket,
    COUNT(*) AS n
FROM bucketed
GROUP BY 1, 2
ORDER BY
    CASE position_bucket
        WHEN '1-2.9' THEN 1
        WHEN '3-5.9' THEN 2
        WHEN '6-10.9' THEN 3
        WHEN '11-20.9' THEN 4
        ELSE 5
    END,
    CASE ctr_bucket
        WHEN '<1%' THEN 1
        WHEN '1-3%' THEN 2
        WHEN '3-5%' THEN 3
        ELSE 4
    END
""").show()

SIGNAL 1 — STALENESS BUCKETS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────┬────────┐
│ staleness_bucket │   n    │  pct   │
│     varchar      │ int64  │ double │
├──────────────────┼────────┼────────┤
│ <7 days          │ 409205 │  95.77 │
│ 30-59 days       │    121 │   0.03 │
│ 60+ days         │  17966 │    4.2 │
└──────────────────┴────────┴────────┘

SIGNAL 2 — CTR BY POSITION BUCKET


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────┬─────────┐
│ position_bucket │ ctr_bucket │    n    │
│     varchar     │  varchar   │  int64  │
├─────────────────┼────────────┼─────────┤
│ 1-2.9           │ <1%        │ 4410147 │
│ 1-2.9           │ 1-3%       │  165438 │
│ 1-2.9           │ 3-5%       │   41292 │
│ 1-2.9           │ 5%+        │   72790 │
│ 3-5.9           │ <1%        │ 4849022 │
│ 3-5.9           │ 1-3%       │  365143 │
│ 3-5.9           │ 3-5%       │   81607 │
│ 3-5.9           │ 5%+        │  101440 │
│ 6-10.9          │ <1%        │ 6570636 │
│ 6-10.9          │ 1-3%       │  312758 │
│ 6-10.9          │ 3-5%       │   81571 │
│ 6-10.9          │ 5%+        │  101888 │
│ 11-20.9         │ <1%        │ 3822672 │
│ 11-20.9         │ 1-3%       │  139417 │
│ 11-20.9         │ 3-5%       │   47556 │
│ 11-20.9         │ 5%+        │   67086 │
│ 21+             │ <1%        │ 7591935 │
│ 21+             │ 1-3%       │   67533 │
│ 21+             │ 3-5%       │   23385 │
│ 21+      

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# ============================================================
# SECTION 2 — BUILD THE RANKED QUEUE
# ============================================================

# We use the latest available date as the decision date.
# No dates after this point are used.

import os

queue = con.sql("""
WITH latest AS (
    SELECT MAX(report_date) AS decision_date
    FROM daily
),

-- Last observed performance date for each content item.
content_last_seen AS (
    SELECT
        content_hash_id,
        MAX(report_date) AS last_observed
    FROM daily
    GROUP BY content_hash_id
),

-- Aggregate performance on the latest observed day for each
-- content item.
latest_content AS (
    SELECT
        d.content_hash_id,
        MAX(d.client_hash_id) AS client_hash_id,
        MAX(d.gsc_impressions) AS gsc_impressions,
        MAX(d.gsc_clicks) AS gsc_clicks,
        MAX(d.gsc_avg_position) AS gsc_avg_position
    FROM daily d
    INNER JOIN latest l
        ON d.report_date = l.decision_date
    GROUP BY d.content_hash_id
),

-- Position buckets provide the comparison group for CTR.
positioned AS (
    SELECT
        content_hash_id,
        client_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_avg_position < 3 THEN '1-2.9'
            WHEN gsc_avg_position < 6 THEN '3-5.9'
            WHEN gsc_avg_position < 11 THEN '6-10.9'
            WHEN gsc_avg_position < 21 THEN '11-20.9'
            ELSE '21+'
        END AS position_bucket,

        CASE
            WHEN gsc_impressions > 0
                THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS ctr

    FROM latest_content
),

-- Establish the typical CTR for each position bucket.
position_medians AS (
    SELECT
        position_bucket,
        MEDIAN(ctr) AS median_ctr
    FROM positioned
    WHERE ctr IS NOT NULL
    GROUP BY position_bucket
),

-- Combine the two signals.
scored AS (
    SELECT
        p.content_hash_id,
        p.client_hash_id,
        p.gsc_impressions,
        p.gsc_clicks,
        p.gsc_avg_position,
        p.position_bucket,
        p.ctr,
        pm.median_ctr,

        DATE_DIFF(
            'day',
            cls.last_observed,
            l.decision_date
        ) AS days_stale,

        CASE
            WHEN DATE_DIFF(
                'day',
                cls.last_observed,
                l.decision_date
            ) < 7 THEN 0

            WHEN DATE_DIFF(
                'day',
                cls.last_observed,
                l.decision_date
            ) < 30 THEN 1

            WHEN DATE_DIFF(
                'day',
                cls.last_observed,
                l.decision_date
            ) < 60 THEN 2

            ELSE 3
        END AS staleness_points,

        CASE
            WHEN p.ctr IS NOT NULL
             AND pm.median_ctr IS NOT NULL
             AND p.ctr < pm.median_ctr
                THEN 2
            ELSE 0
        END AS ctr_points

    FROM positioned p

    LEFT JOIN position_medians pm
        ON p.position_bucket = pm.position_bucket

    INNER JOIN content_last_seen cls
        ON p.content_hash_id = cls.content_hash_id

    CROSS JOIN latest l
)

SELECT
    content_hash_id,
    client_hash_id,

    gsc_impressions AS impressions,
    gsc_clicks AS clicks,
    ROUND(ctr * 100, 3) AS ctr_pct,
    ROUND(median_ctr * 100, 3) AS position_median_ctr_pct,
    ROUND(gsc_avg_position, 2) AS avg_position,

    days_stale,
    staleness_points,
    ctr_points,

    staleness_points + ctr_points AS score,

    CASE
        WHEN staleness_points > 0
         AND ctr_points > 0
            THEN 'REFRESH_OR_CTR'

        WHEN staleness_points > 0
            THEN 'STALE_REFRESH'

        WHEN ctr_points > 0
            THEN 'LOW_CTR_FOR_POSITION'

        ELSE 'MONITOR'
    END AS reason_code,

    CASE
        WHEN staleness_points + ctr_points >= 3
            THEN 'REVIEW'

        ELSE 'MONITOR'
    END AS action

FROM scored
ORDER BY
    score DESC,
    days_stale DESC,
    content_hash_id
""").df()


# Add an explicit rank.
queue.insert(
    0,
    "rank",
    range(1, len(queue) + 1)
)

# Make sure the output directory exists.
os.makedirs("work/outputs", exist_ok=True)

# Write the required CSV.
output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)

print(f"Wrote {output_path}")
print(f"Rows written: {len(queue):,}")

print("\nTop 10:")
display(queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote work/outputs/baseline_action_score.csv
Rows written: 339,652

Top 10:


,rank,content_hash_id,client_hash_id,impressions,clicks,ctr_pct,position_median_ctr_pct,avg_position,days_stale,staleness_points,ctr_points,score,reason_code,action
0,1,content_000005d4ced12088,client_9958f0a7ae1df715,6,0,0.0,0.0,83.67,0,0,0,0,MONITOR,MONITOR
1,2,content_00000c99413ae2ad,client_7de9989c909e91a5,21,0,0.0,0.0,4.52,0,0,0,0,MONITOR,MONITOR
2,3,content_00001e488b74b799,client_625b6439094e23e4,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR
3,4,content_00007bd2985b77c3,client_73cda7b4e4f265ea,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR
4,5,content_00008950670cb6b5,client_def0955f7a377868,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR
5,6,content_0000a348850eb1fc,client_3ffa76342f366962,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR
6,7,content_0000c57e204651e5,client_3ffa76342f366962,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR
7,8,content_0000cd28fbda69f3,client_3ffa76342f366962,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR
8,9,content_0000d31f3926ea12,client_3ffa76342f366962,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR
9,10,content_0000d495bfbfb4a8,client_2094c6eb080311d5,0,0,NaN,0.0,NaN,0,0,0,0,MONITOR,MONITOR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 3 — TOP-20 REVIEW
# ============================================================

top20 = queue.head(20).copy()

def confidence_note(row):
    notes = []

    if row["staleness_points"] > 0:
        notes.append(
            f"{int(row['days_stale'])} days since last observation"
        )

    if row["ctr_points"] > 0:
        if pd.notna(row["ctr_pct"]) and pd.notna(row["position_median_ctr_pct"]):
            notes.append(
                f"CTR {row['ctr_pct']:.2f}% is below "
                f"position median {row['position_median_ctr_pct']:.2f}%"
            )

    if not notes:
        notes.append("no strong signal")

    return "; ".join(notes)


def what_would_make_it_wrong(row):
    reasons = []

    if row["staleness_points"] > 0:
        reasons.append(
            "the content may be intentionally evergreen or not need a refresh"
        )

    if row["ctr_points"] > 0:
        reasons.append(
            "CTR may be noisy because impressions or clicks are limited"
        )

    if not reasons:
        reasons.append(
            "the observed signals may not represent a real action opportunity"
        )

    return "; ".join(reasons)


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

print("\nTop-20 rows:", len(review))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,rank,content_hash_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_000005d4ced12088,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
1,2,content_00000c99413ae2ad,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
2,3,content_00001e488b74b799,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
3,4,content_00007bd2985b77c3,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
4,5,content_00008950670cb6b5,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
5,6,content_0000a348850eb1fc,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
6,7,content_0000c57e204651e5,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
7,8,content_0000cd28fbda69f3,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
8,9,content_0000d31f3926ea12,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...
9,10,content_0000d495bfbfb4a8,MONITOR,MONITOR,0,no strong signal,the observed signals may not represent a real ...



Top-20 rows: 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# ============================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# ============================================================
import pandas as pd

print("WEAK / QUESTIONABLE PICKS")
print("-------------------------")

# Low-confidence high-score rows:
# high score but relatively little search exposure.
weak_picks = queue[
    (queue["score"] >= 3) &
    (queue["impressions"] < 10)
].head(10)

if len(weak_picks) == 0:
    print("No high-score rows with fewer than 10 impressions were found.")
else:
    display(
        weak_picks[
            [
                "rank",
                "content_hash_id",
                "score",
                "reason_code",
                "action",
                "impressions",
                "ctr_pct",
                "avg_position",
                "days_stale"
            ]
        ]
    )

# ------------------------------------------------------------
# Leakage checks
# ------------------------------------------------------------

print("\nLEAKAGE CHECK")
print("-------------")

# Columns that would be suspicious product/label context.
product_flag_names = {
    "health_score",
    "needs_indexing",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate",
    "recommended_action",
    "action_type"
}

used_columns = set(queue.columns)

flag_overlap = used_columns.intersection(product_flag_names)

print("Product-flag columns used in queue:", flag_overlap)

assert len(flag_overlap) == 0, (
    "Leakage check failed: product flag columns were included."
)

# Check that the ranking is actually descending.
assert queue["score"].is_monotonic_decreasing, (
    "Ranking check failed: score is not descending."
)

# Check that ranks are unique and consecutive.
assert queue["rank"].equals(
    pd.Series(range(1, len(queue) + 1))
), "Rank check failed."

# Check the required output exists.
assert os.path.exists(
    "work/outputs/baseline_action_score.csv"
), "CSV was not created."

print("✓ No product-flag columns used.")
print("✓ Scores are ranked descending.")
print("✓ Ranks are consecutive.")
print("✓ CSV exists.")
print("✓ Baseline uses observed signals only.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


WEAK / QUESTIONABLE PICKS
-------------------------
No high-score rows with fewer than 10 impressions were found.

LEAKAGE CHECK
-------------
Product-flag columns used in queue: set()
✓ No product-flag columns used.
✓ Scores are ranked descending.
✓ Ranks are consecutive.
✓ CSV exists.
✓ Baseline uses observed signals only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.